# SQuAD - BERT Fine-Tuning

## Objective

This notebook trains BERT for extractive Question Answering using the SQuAD v1.1 dataset.

Two approaches are evaluated:

- Full Fine-Tuning: all BERT parameters are trainable.
- Feature-Based: BERT parameters are frozen and only the question answering head is trained.

The base model is `bert-base-uncased`.

A fixed random seed of 42 is used for reproducibility.


DRIVE

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Reproducibility and Configuration

Google Drive is used to store checkpoints and trained models.

A fixed random seed of 42 is used throughout the experiment.

Both approaches use the same pretrained model, `bert-base-uncased`.

The trained models and checkpoints are stored in Google Drive to preserve the results between Colab sessions.

CARPETAS

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/BERT_FineTuning_Project"

os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/squad_prepared", exist_ok=True)

print("Carpetas listas.")

Carpetas listas.


LIBRERIAS

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate

SEMILLA

In [ ]:
SEED = 42

import os
import random
import numpy as np
import torch
from transformers import set_seed

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print("Seed configurada:", SEED)

Seed configurada: 42


DATASET

## Dataset: SQuAD v1.1

The Stanford Question Answering Dataset (SQuAD v1.1) is used for extractive question answering.

Each example contains:

- A question
- A context paragraph
- An answer located inside the context

The model must predict the start and end positions of the answer within the context.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


EJEMPLOS 15000

In [ ]:
train_dataset = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(15000))
)

validation_dataset = dataset["validation"]

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))

Train: 15000
Validation: 10570


## Dataset Preparation

The training data is limited to 15,000 examples for this experiment.

The validation set is kept available for evaluation.

The training examples are shuffled using the fixed seed of 42.

In [ ]:
print(train_dataset[0])

{'id': '573173d8497a881900248f0c', 'title': 'Egypt', 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.', 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?', 'answers': {'text': ['84%'], 'answer_start': [468]}}


TOKENIZER

## Tokenization and Answer Alignment

The question and context are tokenized using the BERT tokenizer.

A maximum sequence length of 384 tokens is used, with a document stride of 128 tokens.

Because a context can be longer than the maximum sequence length, the document stride allows overlapping sections of the context to be processed.

The answer positions are aligned with the resulting token positions so the model can learn to predict the beginning and end of the answer.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer cargado correctamente.")

Tokenizer cargado correctamente.


PARAMETROS

In [ ]:
MAX_LENGTH = 384
DOC_STRIDE = 128

print("MAX_LENGTH:", MAX_LENGTH)
print("DOC_STRIDE:", DOC_STRIDE)

MAX_LENGTH: 384
DOC_STRIDE: 128


E

In [ ]:
def prepare_train_features(examples):
    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(input_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        if (
            offsets[context_start][0] > start_char
            or offsets[context_end][1] < end_char
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:
            token_start_index = context_start

            while (
                token_start_index <= context_end
                and offsets[token_start_index][0] <= start_char
            ):
                token_start_index += 1

            start_positions.append(token_start_index - 1)

            token_end_index = context_end

            while (
                token_end_index >= context_start
                and offsets[token_end_index][1] >= end_char
            ):
                token_end_index -= 1

            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [ ]:
tokenized_train = train_dataset.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(tokenized_train)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 15141
})


In [ ]:
def prepare_validation_features(examples):

    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")

    tokenized["example_id"] = []

    new_offset_mapping = []

    for i in range(len(tokenized["input_ids"])):

        sequence_ids = tokenized.sequence_ids(i)
        offsets = tokenized["offset_mapping"][i]

        sample_index = sample_mapping[i]

        tokenized["example_id"].append(
            examples["id"][sample_index]
        )

        new_offsets = []

        for k, offset in enumerate(offsets):
            if sequence_ids[k] == 1:
                new_offsets.append(offset)
            else:
                new_offsets.append(None)

        new_offset_mapping.append(new_offsets)

    tokenized["offset_mapping"] = new_offset_mapping

    return tokenized

In [ ]:
tokenized_validation = validation_dataset.map(
    prepare_validation_features,
    batched=True,
    remove_columns=validation_dataset.column_names
)

print(tokenized_validation)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 10753
})


In [ ]:
TRAIN_FEATURES_DIR = f"{BASE_DIR}/squad_prepared/train"
VALIDATION_FEATURES_DIR = f"{BASE_DIR}/squad_prepared/validation"

tokenized_train.save_to_disk(TRAIN_FEATURES_DIR)
tokenized_validation.save_to_disk(VALIDATION_FEATURES_DIR)

print("Datos preparados y guardados en Google Drive.")

Saving the dataset (0/1 shards):   0%|          | 0/15141 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10753 [00:00<?, ? examples/s]

Datos preparados y guardados en Google Drive.


## Full Fine-Tuning

In the Full Fine-Tuning approach, all BERT parameters are trainable.

The model can adapt its pretrained representations specifically to the question answering task.

The learning rate used is `2e-5`.

MOD 1

In [ ]:
from transformers import AutoModelForQuestionAnswering

model_full = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
)

print("Modelo Full Fine-tuning cargado.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Modelo Full Fine-tuning cargado.


In [ ]:
total_params = sum(
    p.numel() for p in model_full.parameters()
)

trainable_params = sum(
    p.numel() for p in model_full.parameters()
    if p.requires_grad
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)

Parámetros totales: 108893186
Parámetros entrenables: 108893186


In [ ]:
from transformers import TrainingArguments

training_args_full = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/squad_full",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=False
)

print("Configuración Full preparada.")

Configuración Full preparada.


In [ ]:
from transformers import Trainer

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=tokenized_train,
    processing_class=tokenizer
)

print("Trainer Full preparado.")

Trainer Full preparado.


In [ ]:
trainer_full.train()

Step,Training Loss
100,4.074638
200,2.563487
300,2.320021
400,2.128634
500,1.926965
600,1.803211
700,1.630525
800,1.669321
900,1.503174


Step,Training Loss
100,4.074638
200,2.563487
300,2.320021
400,2.128634
500,1.926965
600,1.803211
700,1.630525
800,1.669321
900,1.503174
1000,1.605166


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3786, training_loss=1.3212609893105614, metrics={'train_runtime': 2199.1928, 'train_samples_per_second': 13.77, 'train_steps_per_second': 1.722, 'total_flos': 5934441290609664.0, 'train_loss': 1.3212609893105614, 'epoch': 2.0})

MODELO 2

## Feature-Based Training

In the Feature-Based approach, the pretrained BERT parameters are frozen.

Only the question answering head is trained.

This allows the experiment to evaluate how well the pretrained BERT representations perform when they are used as fixed features.

In [ ]:
model_feature = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
)

print("Modelo Feature-based cargado.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Modelo Feature-based cargado.


In [ ]:
for param in model_feature.base_model.parameters():
    param.requires_grad = False

for param in model_feature.qa_outputs.parameters():
    param.requires_grad = True

total_params = sum(
    p.numel() for p in model_feature.parameters()
)

trainable_params = sum(
    p.numel() for p in model_feature.parameters()
    if p.requires_grad
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)
print("BERT congelado correctamente.")

Parámetros totales: 108893186
Parámetros entrenables: 1538
BERT congelado correctamente.


In [ ]:
training_args_feature = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/squad_feature",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=False
)

print("Configuración Feature-based preparada.")

Configuración Feature-based preparada.


In [ ]:
trainer_feature = Trainer(
    model=model_feature,
    args=training_args_feature,
    train_dataset=tokenized_train,
    processing_class=tokenizer
)

print("Trainer Feature-based preparado.")

Trainer Feature-based preparado.


## Evaluation

The models are evaluated using two standard extractive question answering metrics:

- Exact Match (EM): measures whether the predicted answer exactly matches the reference answer.
- F1 Score: measures the overlap between the predicted answer and the reference answer.

The evaluation uses the predicted answer spans after the question answering post-processing step.

In [ ]:
trainer_feature.train()

Step,Training Loss
100,4.999827
200,4.219321
300,3.988228
400,3.858215
500,3.785764
600,3.780110
700,3.678180
800,3.684007
900,3.598476
1000,3.637820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3786, training_loss=3.629620777257082, metrics={'train_runtime': 748.041, 'train_samples_per_second': 40.482, 'train_steps_per_second': 5.061, 'total_flos': 5934441290609664.0, 'train_loss': 3.629620777257082, 'epoch': 2.0})

In [ ]:
MODEL_DIR_FULL = f"{BASE_DIR}/models/squad_bert_qa_full"

trainer_full.save_model(MODEL_DIR_FULL)
tokenizer.save_pretrained(MODEL_DIR_FULL)

print("Modelo Full Fine-tuning guardado en:")
print(MODEL_DIR_FULL)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo Full Fine-tuning guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/squad_bert_qa_full


In [ ]:
!ls -lah "$BASE_DIR/models/squad_bert_qa_full"

total 417M
drwx------ 2 root root 4.0K Sep 25 10:03 .
drwx------ 3 root root 4.0K Sep 25 10:03 ..
-rw------- 1 root root  762 Sep 25 10:03 config.json
-rw------- 1 root root 416M Sep 25 10:03 model.safetensors
-rw------- 1 root root  351 Sep 25 10:03 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 10:03 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 10:03 training_args.bin


In [ ]:
MODEL_DIR_FEATURE = f"{BASE_DIR}/models/squad_bert_qa_feature"

trainer_feature.save_model(MODEL_DIR_FEATURE)
tokenizer.save_pretrained(MODEL_DIR_FEATURE)

print("Modelo Feature-based guardado en:")
print(MODEL_DIR_FEATURE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo Feature-based guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/squad_bert_qa_feature


In [ ]:
!ls -lah "$BASE_DIR/models/squad_bert_qa_feature"

total 417M
drwx------ 2 root root 4.0K Sep 25 10:04 .
drwx------ 4 root root 4.0K Sep 25 10:04 ..
-rw------- 1 root root  762 Sep 25 10:04 config.json
-rw------- 1 root root 416M Sep 25 10:04 model.safetensors
-rw------- 1 root root  351 Sep 25 10:04 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 10:04 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 10:04 training_args.bin


In [ ]:
import evaluate

squad_metric = evaluate.load("squad")

print("Métrica SQuAD cargada correctamente.")

Métrica SQuAD cargada correctamente.


In [ ]:
import numpy as np

def postprocess_qa_predictions(
    examples,
    features,
    predictions,
    n_best_size=20,
    max_answer_length=30
):
    all_start_logits, all_end_logits = predictions

    example_id_to_index = {
        k: i for i, k in enumerate(examples["id"])
    }

    features_per_example = {}

    for feature_index, example_id in enumerate(features["example_id"]):
        if example_id not in features_per_example:
            features_per_example[example_id] = []

        features_per_example[example_id].append(feature_index)

    predictions_dict = {}

    for example_id, example_index in example_id_to_index.items():

        feature_indices = features_per_example.get(example_id, [])

        context = examples[example_index]["context"]

        valid_answers = []

        for feature_index in feature_indices:

            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]

            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-n_best_size:][::-1]
            end_indexes = np.argsort(end_logits)[-n_best_size:][::-1]

            for start_index in start_indexes:

                if start_index >= len(offsets):
                    continue

                if offsets[start_index] is None:
                    continue

                for end_index in end_indexes:

                    if end_index >= len(offsets):
                        continue

                    if offsets[end_index] is None:
                        continue

                    if end_index < start_index:
                        continue

                    if end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    if start_char is None or end_char is None:
                        continue

                    text = context[start_char:end_char]

                    score = (
                        float(start_logits[start_index])
                        + float(end_logits[end_index])
                    )

                    valid_answers.append({
                        "text": text,
                        "score": score
                    })

        if valid_answers:
            best_answer = max(
                valid_answers,
                key=lambda x: x["score"]
            )

            predictions_dict[example_id] = best_answer["text"]

        else:
            predictions_dict[example_id] = ""

    formatted_predictions = [
        {
            "id": example_id,
            "prediction_text": predictions_dict[example_id]
        }
        for example_id in examples["id"]
    ]

    references = [
        {
            "id": example["id"],
            "answers": example["answers"]
        }
        for example in examples
    ]

    return formatted_predictions, references

In [ ]:
predictions_full = trainer_full.predict(tokenized_validation)

print("Predicciones Full Fine-tuning obtenidas.")

Predicciones Full Fine-tuning obtenidas.


In [ ]:
formatted_predictions_full, references = postprocess_qa_predictions(
    validation_dataset,
    tokenized_validation,
    predictions_full.predictions
)

results_full_qa = squad_metric.compute(
    predictions=formatted_predictions_full,
    references=references
)

print("SQuAD Full Fine-tuning")
print("Exact Match:", results_full_qa["exact_match"])
print("F1:", results_full_qa["f1"])

SQuAD Full Fine-tuning
Exact Match: 72.53547776726585
F1: 81.74745562989544


In [ ]:
predictions_feature = trainer_feature.predict(tokenized_validation)

print("Predicciones Feature-based obtenidas.")

Predicciones Feature-based obtenidas.


In [ ]:
formatted_predictions_feature, references = postprocess_qa_predictions(
    validation_dataset,
    tokenized_validation,
    predictions_feature.predictions
)

results_feature_qa = squad_metric.compute(
    predictions=formatted_predictions_feature,
    references=references
)

print("SQuAD Feature-based")
print("Exact Match:", results_feature_qa["exact_match"])
print("F1:", results_feature_qa["f1"])

SQuAD Feature-based
Exact Match: 15.383159886471145
F1: 24.234021285001567


In [ ]:
import json
import os

results_qa = {
    "full_fine_tuning": {
        "exact_match": results_full_qa["exact_match"],
        "f1": results_full_qa["f1"]
    },
    "feature_based": {
        "exact_match": results_feature_qa["exact_match"],
        "f1": results_feature_qa["f1"]
    }
}

results_path = f"{BASE_DIR}/squad_results.json"

with open(results_path, "w") as f:
    json.dump(results_qa, f, indent=4)

print("Resultados guardados en:")
print(results_path)

Resultados guardados en:
/content/drive/MyDrive/BERT_FineTuning_Project/squad_results.json


## Final Results

### Full Fine-Tuning

- Exact Match: 72.54%
- F1 Score: 81.75%

### Feature-Based

- Exact Match: 15.38%
- F1 Score: 24.23%

Full Fine-Tuning achieved substantially higher performance because the BERT parameters were allowed to adapt to the question answering task.

## Reproducibility Note

The experiment uses `bert-base-uncased`, random seed 42, a maximum sequence length of 384, and a document stride of 128.

The training set was limited to 15,000 examples.

Training was performed in Google Colab using GPU acceleration, and trained models were saved to Google Drive.